# 207. LLM 水印：绿色名单采样与 z-score 检测怎样实现？

> **面试问题：怎样从密钥和前缀生成 green list、偏置采样分布、计算统计检验，并处理短文本、编辑攻击、阈值和密钥轮换？**

## 先给结论

这里的关键不是调用一个安全/训练/推理框架，而是定义输入、状态、不变量、失败分支和独立的判断 oracle。下方仅以受控小数据验证机制；真实服务仍需替换模型、密钥管理、访问控制、审计、红队评测和线上 SLO。

## 一手资料

- [A Watermark for Large Language Models](https://arxiv.org/abs/2301.10226)
- [Certified Robust Watermark](https://arxiv.org/abs/2409.19708)
- [Watermarking Survey](https://arxiv.org/abs/2312.07969)

In [ ]:
notebook_contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "security-and-versioning-required"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：水印是在采样分布中嵌入可统计检验的信号

绿色名单水印不是给文本附加肉眼可见标签，而是使用密钥和前缀为下一 token 划分 green/red 集，并轻微提升 green token 的采样分数。检测方统计 green 命中数；密钥、词表、tokenizer 和上下文规则必须版本一致。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
vocab = ["春", "夏", "秋", "冬", "风", "雨", "云", "山"]  # 执行本行的状态、计算或校验逻辑。
key_id = "wm-key-v1"  # 执行本行的状态、计算或校验逻辑。
assert len(vocab) == 8  # 执行本行的状态、计算或校验逻辑。
assert len(set(vocab)) == len(vocab)  # 执行本行的状态、计算或校验逻辑。
assert key_id == "wm-key-v1"  # 执行本行的状态、计算或校验逻辑。


## 2. 绿色名单：由密钥和前缀确定性生成

同一个前缀在同一密钥/词表下必须得到相同 green list，才能让生成和检测一致。教学实现用哈希排序选择一半词；生产需使用保密密钥、密码学 PRF、明确的 context window 与 key rotation。


In [ ]:
def green_list(prefix, vocab, key_id, fraction=0.5):  # 执行本行的状态、计算或校验逻辑。
    ranked = sorted(vocab, key=lambda token: hashlib.sha256(f"{key_id}|{prefix}|{token}".encode()).hexdigest())  # 执行本行的状态、计算或校验逻辑。
    return set(ranked[:max(1, int(len(vocab) * fraction))])  # 执行本行的状态、计算或校验逻辑。
green = green_list("春风", vocab, key_id)  # 执行本行的状态、计算或校验逻辑。
assert len(green) == 4  # 执行本行的状态、计算或校验逻辑。
assert green == green_list("春风", vocab, key_id)  # 执行本行的状态、计算或校验逻辑。
assert green != green_list("秋雨", vocab, key_id)  # 执行本行的状态、计算或校验逻辑。


## 3. 采样偏置：只改变 green token 的 logit

水印应在模型已经得到 logits 后施加小偏置，而不重写模型权重。下面实现稳定 softmax；实际服务要在 temperature、top-p、grammar mask 和安全过滤之后定义清楚水印施加顺序。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
logits = {"春": 1.0, "夏": 0.8, "秋": 0.7, "冬": 0.6, "风": 0.5, "雨": 0.4, "云": 0.3, "山": 0.2}  # 执行本行的状态、计算或校验逻辑。
def apply_green_bias(logits, green, delta):  # 执行本行的状态、计算或校验逻辑。
    return {token: value + (delta if token in green else 0.0) for token, value in logits.items()}  # 执行本行的状态、计算或校验逻辑。
def softmax(values):  # 执行本行的状态、计算或校验逻辑。
    maximum = max(values.values())  # 执行本行的状态、计算或校验逻辑。
    exps = {key: math.exp(value - maximum) for key, value in values.items()}  # 执行本行的状态、计算或校验逻辑。
    total = sum(exps.values())  # 执行本行的状态、计算或校验逻辑。
    return {key: value / total for key, value in exps.items()}  # 执行本行的状态、计算或校验逻辑。
biased_probs = softmax(apply_green_bias(logits, green, 1.0))  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(biased_probs.values()), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert sum(biased_probs[token] for token in green) > 0.5  # 执行本行的状态、计算或校验逻辑。
assert set(biased_probs) == set(vocab)  # 执行本行的状态、计算或校验逻辑。


## 4. 生成：记录 token 及其生成前缀

检测必须使用每个 token 生成时的前缀来重建 green list。这里以固定选择序列模拟已生成文本；生产生成仍受随机种子、采样器、stop token 和 streaming 分块影响，均需纳入 trace。


In [ ]:
generated = ["春", "风", "云", "山", "夏", "雨"]  # 执行本行的状态、计算或校验逻辑。
def green_hits(tokens, key_id):  # 执行本行的状态、计算或校验逻辑。
    hits = 0  # 执行本行的状态、计算或校验逻辑。
    for index, token in enumerate(tokens):  # 执行本行的状态、计算或校验逻辑。
        prefix = "".join(tokens[:index])  # 执行本行的状态、计算或校验逻辑。
        hits += int(token in green_list(prefix, vocab, key_id))  # 执行本行的状态、计算或校验逻辑。
    return hits  # 执行本行的状态、计算或校验逻辑。
hit_count = green_hits(generated, key_id)  # 执行本行的状态、计算或校验逻辑。
assert 0 <= hit_count <= len(generated)  # 执行本行的状态、计算或校验逻辑。
assert green_hits(generated, key_id) == hit_count  # 执行本行的状态、计算或校验逻辑。
assert len(generated) == 6  # 执行本行的状态、计算或校验逻辑。


## 5. 检测：用 z-score 比较实际 green 命中与零假设

若零假设下每 token 进入 green list 的概率为 gamma，z-score 是 `(g-gamma*n)/sqrt(n*gamma*(1-gamma))`。这是统计证据，不是来源的绝对证明；短文本、编辑攻击和错误 tokenizer 都会削弱检验。


In [ ]:
def watermark_z_score(hits, count, gamma=0.5):  # 执行本行的状态、计算或校验逻辑。
    if count <= 0 or not 0 < gamma < 1:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("检测长度和绿色比例必须有效")  # 执行本行的状态、计算或校验逻辑。
    return (hits - gamma * count) / math.sqrt(count * gamma * (1 - gamma))  # 执行本行的状态、计算或校验逻辑。
z_value = watermark_z_score(hit_count, len(generated))  # 执行本行的状态、计算或校验逻辑。
assert math.isfinite(z_value)  # 执行本行的状态、计算或校验逻辑。
assert watermark_z_score(6, 6) > 2.0  # 执行本行的状态、计算或校验逻辑。
assert watermark_z_score(3, 6) == 0.0  # 执行本行的状态、计算或校验逻辑。


## 6. 判定：阈值需要控制误报与漏报

高 z 值支持“文本与该水印机制相符”，不是司法意义的归因。阈值应根据文本长度、目标 FPR、攻击模型和语言分桶校准；输出应包含 inconclusive 状态而不是硬二分。


In [ ]:
def detect(z_value, threshold):  # 执行本行的状态、计算或校验逻辑。
    return "watermark_supported" if z_value >= threshold else "inconclusive"  # 执行本行的状态、计算或校验逻辑。
assert detect(2.5, 2.0) == "watermark_supported"  # 执行本行的状态、计算或校验逻辑。
assert detect(1.9, 2.0) == "inconclusive"  # 执行本行的状态、计算或校验逻辑。
assert detect(0.0, 2.0) == "inconclusive"  # 执行本行的状态、计算或校验逻辑。


## 7. 攻击与失败：编辑、翻译和短文本会降低检测力

攻击者可替换、删改或改写 token，合法用户也可能产生短输出。必须在保留语义的攻击集、不同长度和不同解码设置上测 ROC/FPR；不能仅报告未编辑样本文本的命中率。


In [ ]:
def edited_tokens(tokens):  # 执行本行的状态、计算或校验逻辑。
    return ["秋" if token == "春" else token for token in tokens]  # 执行本行的状态、计算或校验逻辑。
edited = edited_tokens(generated)  # 执行本行的状态、计算或校验逻辑。
assert len(edited) == len(generated)  # 执行本行的状态、计算或校验逻辑。
assert edited[0] == "秋"  # 执行本行的状态、计算或校验逻辑。
assert green_hits(edited, key_id) != hit_count or edited != generated  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：检测配置不能泄露密钥本身

审计可以保存 key id、算法版本、tokenizer、context width、gamma、delta、阈值和 z-score，但不应写入原始 secret。密钥轮换后应保留受限的历史验证能力，并明确每把 key 的生效区间。


In [ ]:
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"key_id": key_id, "algorithm": "greenlist-v1", "tokenizer": "tok-v1", "gamma": 0.5, "threshold": 2.0, "z": z_value}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert "secret" not in artifact  # 执行本行的状态、计算或校验逻辑。
assert artifact["key_id"] == "wm-key-v1"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整回答应覆盖目标、显式数据结构、核心规则、边界失败、评测指标和版本制品。受控断言只验证实现不变量，不能被解读为真实模型质量、攻击鲁棒性、隐私合规或线上成本结论。
